# Function Tool

Custom code that agents can call during conversations

In [1]:
import os
import inspect
import typing
import types
import pprint
from agent_framework import (
    tool,
    FunctionTool,
    FunctionInvocationContext,
    Agent
)
from agent_framework.openai import OpenAIChatClient
from typing import Never, Annotated, TypedDict
from pydantic import Field, BaseModel
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
chat_client = OpenAIChatClient(
    base_url=os.getenv("FOUNDRY_OPENAI_BASE_URL"),
    api_key=os.getenv("FOUNDRY_API_KEY"),
    model=os.getenv("FOUNDRY_MODEL"),
)

## Inspect Function Tool

Note: You can turn any Python function into a function tool by passing it to the agent's tools parameter when creating the agent. You do not need the `@tool` decorator.

In [3]:
# Note: if you don't specify the name and description parameters in a @tool decorator, the framework will automatically use the function's name and docstring as fallbacks.
def get_previous_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")], # required
    ctx: FunctionInvocationContext, # automatically detected by framework and marked as not required
    city: Annotated[str, Field(description="The city to get the weather for.")] = "Halifax", # not required
) -> str:
    """Get the previous weather for a given location."""
    return f"The previous weather in {location} was sunny with a high of 30°C."

@tool(name="weather_tool", description="Get the weather for a location", approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")], # required
    ctx: FunctionInvocationContext, # automatically detected by framework and marked as not required
    city: Annotated[str, Field(description="The city to get the weather for.")] = "Halifax", # not required
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [4]:
from typing import TypedDict

class ToolParameterInfo(TypedDict):
    name: str
    type: list[type]
    required: bool
    default: object | None

class ToolInfo(TypedDict):
    name: str
    description: str
    approval_mode: str
    parameters: list[ToolParameterInfo]
    return_type: list[type]

In [5]:
def is_function_tool(func) -> bool:
    return isinstance(func, FunctionTool)

def _extract_types(tp) -> list[type]:
    if tp is None:
        return []
    
    origin = typing.get_origin(tp)
    if origin in (typing.Union, types.UnionType):
        return [t for t in typing.get_args(tp)]
    return [tp]

def get_functiontool_return_type(tool: FunctionTool) -> list[type]:
    if not is_function_tool(tool):
        raise ValueError("Provided function is not a FunctionTool.")
    
    hints = typing.get_type_hints(tool.func)
    return _extract_types(hints.get("return", None))


def get_functiontool_parameters(tool: FunctionTool) -> list[ToolParameterInfo]:
    if not is_function_tool(tool):
        raise ValueError("Provided function is not a FunctionTool")

    if not tool.func:
        raise ValueError("FunctionTool has no associated function.")
    
    parameters = tool.parameters()
    properties = parameters.get("properties", None)
    if properties is None:
        return []

    required = parameters.get("required", [])

    sig = inspect.signature(tool.func)
    hints = typing.get_type_hints(tool.func)
    parameters_info = []
    
    for param_name, param in sig.parameters.items():
        param_type = hints.get(param_name, None)
        param_info = ToolParameterInfo(
            name=param_name,
            type=_extract_types(param_type),
            required=param_name in required,
            default=None if param.default is inspect.Parameter.empty else param.default
        )
        parameters_info.append(param_info)
    
    return parameters_info

def get_functiontool_metadata(tool: FunctionTool) -> ToolInfo:
    if not is_function_tool(tool):
        raise ValueError("Input must be an instance of FunctionTool")
    info: ToolInfo = {
        "name": tool.name,
        "description": tool.description,
        "approval_mode": tool.approval_mode,
        "parameters": get_functiontool_parameters(tool),
        "return_type": get_functiontool_return_type(tool),
    }
    return info


In [6]:
agent = Agent(
    client=chat_client,
    name="Tool agent",
    description="You are a tool agent that can utilize various tools to assist with tasks.",
    instructions="You have access to various tools and should use them to assist with tasks effectively.",
    tools=[get_weather, get_previous_weather],
    context_providers=[],
)

In [7]:
for agent_tool in agent.default_options["tools"]:
    if is_function_tool(agent_tool):
        pprint.pp(get_functiontool_metadata(agent_tool), indent=4)

{   'name': 'weather_tool',
    'description': 'Get the weather for a location',
    'approval_mode': 'never_require',
    'parameters': [   {   'name': 'location',
                          'type': [<class 'str'>],
                          'required': True,
                          'default': None},
                      {   'name': 'ctx',
                          'type': [   <class 'agent_framework._middleware.FunctionInvocationContext'>],
                          'required': False,
                          'default': None},
                      {   'name': 'city',
                          'type': [<class 'str'>],
                          'required': False,
                          'default': 'Halifax'}],
    'return_type': [<class 'str'>]}
{   'name': 'get_previous_weather',
    'description': 'Get the previous weather for a given location.',
    'approval_mode': 'never_require',
    'parameters': [   {   'name': 'location',
                          'type': [<class 'str'>]

## Inspect Class-Based Tool Decorator

In [8]:
class WeatherTools:
    def __init__(self, units: str = 'celcius'):
        self.units = units
    async def get_current_weather(
        self,
        location: Annotated[str, Field(description="The location to get the weather for.")], # required
        ctx: FunctionInvocationContext, # automatically detected by framework and marked as not required
        city: Annotated[str, Field(description="The city to get the weather for.")] = "Halifax", # not required
    ) -> str:
        """Get the weather for a given location."""
        return f"The weather in {location} is cloudy with a high of 15° {self.units[0].upper()}."

    # Become FunctionTool
    async def get_previous_weather(
        self,
        location: Annotated[str, Field(description="The location to get the weather for.")], # required
        ctx: FunctionInvocationContext, # automatically detected by framework and marked as not required
        city: Annotated[str, Field(description="The city to get the weather for.")] = "Halifax", # not required
    ) -> str:
        """Get the previous weather for a given location."""
        return f"The previous weather in {location} was sunny with a high of 30° {self.units[0].upper()}."

    @staticmethod
    def get_units() -> str:
        """Get the units used for weather measurements."""
        return "celcius"

In [ ]:
weather_tools = WeatherTools()

# You cannot use the @tool decorator directly on class methods, so we use the tool() function directly to wrap the class method
get_previous_weather = tool(name="get_old_stuff", description="Get old weather")(weather_tools.get_previous_weather)

agent = Agent(
    client=chat_client,
    name="Tool agent",
    description="You are a tool agent that can utilize various tools to assist with tasks.",
    instructions="You have access to various tools and should use them to assist with tasks effectively.",
    tools=[
        weather_tools.get_current_weather,
        get_previous_weather,
        weather_tools.get_units,
    ],
    context_providers=[],
)

result = await agent.run('What is the current weather and previous weather for Halifax, Nova Scotia?')
print(result)

Here’s the weather information for Halifax, Nova Scotia:

- **Current weather:** Cloudy, with a high of **15°C**  
- **Previous weather:** Sunny, with a high of **30°C**


In [12]:
for tool in agent.default_options["tools"]:
    if is_function_tool(tool):
        pprint.pp(get_functiontool_metadata(tool), indent=4)

{   'name': 'get_current_weather',
    'description': 'Get the weather for a given location.',
    'approval_mode': 'never_require',
    'parameters': [   {   'name': 'location',
                          'type': [<class 'str'>],
                          'required': True,
                          'default': None},
                      {   'name': 'ctx',
                          'type': [   <class 'agent_framework._middleware.FunctionInvocationContext'>],
                          'required': False,
                          'default': None},
                      {   'name': 'city',
                          'type': [<class 'str'>],
                          'required': False,
                          'default': 'Halifax'}],
    'return_type': [<class 'str'>]}
{   'name': 'get_old_stuff',
    'description': 'Get old weather',
    'approval_mode': 'never_require',
    'parameters': [   {   'name': 'location',
                          'type': [<class 'str'>],
                      

## Inspect Schema

## Inspect MCP Tool

## Inspect Agent Function Invocation Configuration

Set limits on the chat client to control automatic tool invocation by model round trips, total function calls, and elapsed wall-clock time

## Inspect Agent as Tool

Scenarios to test:

- @tool()
- @tool(approval_mode, name, description)
- @tool(schema)
- class-based (tool decorator applied to a method)
- class-based (no tool decorator applied)

In [ ]:
# Tool without approval mode
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str | int, Field(description="The location to get the weather for.")],
    ctx: FunctionInvocationContext,
) -> str | int:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [ ]:
print(type(get_weather))

<class 'agent_framework._tools.FunctionTool'>


In [ ]:
dir(get_weather)

['DEFAULT_EXCLUDE',
 'INJECTABLE',
 '_SHALLOW_COPY_FIELDS',
 '__annotations__',
 '__call__',
 '__class__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_cached_parameters',
 '_context_parameter_name',
 '_discover_injected_parameters',
 '_get_type_identifier',
 '_input_model_explicitly_provided',
 '_input_schema',
 '_input_schema_cached',
 '_instance',
 '_invocation_duration_histogram',
 '_is_context_parameter',
 '_make_dumpable',
 '_resolve_input_model',
 '_schema_supplied',
 'additional_properties',
 'approval_mode',
 'declaration_only',
 'description',
 'from_dict',
 '

In [ ]:
# Tool without approval mode
@tool(name="weather_tool", description="Get the weather for a location", approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
    ctx: FunctionInvocationContext,
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [ ]:
class WeatherTool(BaseModel):
    location: Annotated[str, Field(description="The location to get the weather for.")]

# Tool without approval mode
@tool(schema=WeatherTool)
def get_weather_schema(
    location: str,
    ctx: FunctionInvocationContext,
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [ ]:
class MyToolClassWithHandlers:
    @tool(name="weather_tool", description="Get the weather for a location", approval_mode="never_require")
    def get_weather_method(
        self,
        location: Annotated[str, Field(description="The location to get the weather for.")],
        ctx: FunctionInvocationContext,
    ) -> str:
        """Get the weather for a given location."""
        return f"The weather in {location} is cloudy with a high of 15°C."


<class 'agent_framework._tools.FunctionTool'>


In [ ]:
class MyToolClassWithoutHandlers:
    def divide(
        self,
        a: Annotated[float, Field(description="The dividend")],
        b: Annotated[float, Field(description="The divisor")],
        ctx: FunctionInvocationContext
    ) -> float:
        """Divide two numbers."""
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

    def add(
        self,
        a: Annotated[float, Field(description="The first addend")],
        b: Annotated[float, Field(description="The second addend")],
        ctx: FunctionInvocationContext
    ) -> float:
        """Add two numbers."""
        return a + b

In [ ]:
inspect.signature(get_weather)

ValueError: no signature found for builtin <agent_framework._tools.FunctionTool object at 0x0000024FBB90C190>

In [ ]:
get_weather.approval_mode

'never_require'

In [ ]:
get_weather.description

'Get the weather for a given location.'

In [ ]:
get_weather.parameters()

{'properties': {'location': {'anyOf': [{'type': 'string'},
    {'type': 'integer'}],
   'description': 'The location to get the weather for.',
   'title': 'Location'}},
 'required': ['location'],
 'title': 'get_weather_input',
 'type': 'object'}

In [ ]:
inspect.getdoc(get_weather.func)

'Get the weather for a given location.'

In [ ]:
dir(inspect.signature(get_weather.func))

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__replace__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '_bind',
 '_bound_arguments_cls',
 '_hash_basis',
 '_parameter_cls',
 '_parameters',
 '_return_annotation',
 'bind',
 'bind_partial',
 'empty',
 'format',
 'from_callable',
 'parameters',
 'replace',
 'return_annotation']

In [ ]:
inspect.signature(get_weather.func).return_annotation

str | int

In [ ]:
get_weather.to_json()

'{"type": "function_tool", "name": "get_weather", "description": "Get the weather for a given location.", "approval_mode": "never_require", "invocation_count": 0, "invocation_exception_count": 0, "input_model": {"properties": {"location": {"anyOf": [{"type": "string"}, {"type": "integer"}], "description": "The location to get the weather for.", "title": "Location"}}, "required": ["location"], "title": "get_weather_input", "type": "object"}}'

In [ ]:
# This is how we can get the actual types, not just a string representation 
import typing
import types

hints = typing.get_type_hints(get_weather.func)
return_type = hints.get("return", None)

origin = typing.get_origin(return_type)
args = typing.get_args(return_type)

if origin in (typing.Union, types.UnionType):   # handles both forms
    print(args)
else:
    print("Not a union:", return_type)


{'location': <class 'str'>, 'ctx': <class 'agent_framework._middleware.FunctionInvocationContext'>, 'return': str | int}
(<class 'str'>, <class 'int'>)


In [ ]:
# LEFT-OFF: if Gemini suggests to use the string based version, then can we do the same with executors?